### Database Connection Setup

In [1]:
import pymongo
from pymongo import MongoClient

MONGO_URI = "mongodb://localhost:27017"
client = MongoClient(MONGO_URI)

# Access database and reviews collection
db = client["amazon_reviews_db"]
reviews_col = db["reviews"]

print("Connected to database successfully!")

Connected to database successfully!


### Pipeline 1: Average Rating per Product
**Objective:** Identify the highest-rated products in the dataset.

**How it works:** 1. `$group`: Groups all reviews by product ID (`asin`) and calculates the average `overall` rating while keeping a count of total reviews.
2. `$match`: Filters out products with fewer than 5 reviews to ensure statistical relevance (avoiding 5-star items with only 1 review).
3. `$sort` & `$limit`: Orders the results from highest to lowest rating and returns the top 10.

In [2]:
pipeline_avg_rating = [
    {"$group": {
        "_id": "$asin", 
        "avg_rating": {"$avg": "$overall"},
        "total_reviews": {"$sum": 1}
    }},
    {"$match": {"total_reviews": {"$gte": 5}}}, 
    {"$sort": {"avg_rating": -1}},
    {"$limit": 10}
]

print("--- 1. Top Rated Products (Min 5 Reviews) ---")
result_1 = list(reviews_col.aggregate(pipeline_avg_rating))
for doc in result_1:
    print(doc)

--- 1. Top Rated Products (Min 5 Reviews) ---
{'_id': 'B00INCZML4', 'avg_rating': 5.0, 'total_reviews': 6}
{'_id': 'B009S2CXSI', 'avg_rating': 5.0, 'total_reviews': 8}
{'_id': 'B00C46LS9U', 'avg_rating': 5.0, 'total_reviews': 5}
{'_id': 'B01DMHPT3U', 'avg_rating': 5.0, 'total_reviews': 10}
{'_id': 'B00FAYL1YU', 'avg_rating': 5.0, 'total_reviews': 8}
{'_id': 'B0085DZRDE', 'avg_rating': 5.0, 'total_reviews': 6}
{'_id': 'B00KXTZ3BE', 'avg_rating': 5.0, 'total_reviews': 10}
{'_id': 'B000VDCT3C', 'avg_rating': 5.0, 'total_reviews': 5}
{'_id': 'B0016AIYU6', 'avg_rating': 5.0, 'total_reviews': 6}
{'_id': 'B00IKDFL4O', 'avg_rating': 5.0, 'total_reviews': 7}


### Pipeline 2: Top 5 Most Reviewed Products
**Objective:** Identify the most popular/frequently purchased items.

**How it works:** We group the collection by the product ID (`asin`) and use the `$sum` accumulator to add 1 for every document found. We then sort descending by that count to surface the top 5 products.

In [3]:
pipeline_top_products = [
    {"$group": {
        "_id": "$asin", 
        "review_count": {"$sum": 1}
    }},
    {"$sort": {"review_count": -1}},
    {"$limit": 5}
]

print("--- 2. Top 5 Most Reviewed Products ---")
result_2 = list(reviews_col.aggregate(pipeline_top_products))
for doc in result_2:
    print(doc)

--- 2. Top 5 Most Reviewed Products ---
{'_id': 'B003L1ZYYW', 'review_count': 86}
{'_id': 'B0019HL8Q8', 'review_count': 85}
{'_id': 'B0019EHU8G', 'review_count': 74}
{'_id': 'B00IVPU7AO', 'review_count': 73}
{'_id': 'B000BQ7GW8', 'review_count': 71}


### Pipeline 3: Reviews per User (Power Users)
**Objective:** Identify our most active reviewers.

**How it works:** Similar to the product count, this pipeline groups by `reviewerID`. This highlights user engagement and can be used to populate the `total_reviews` field in the User profile embedded document.

In [4]:
pipeline_user_reviews = [
    {"$group": {
        "_id": "$reviewerID", 
        "total_reviews_written": {"$sum": 1}
    }},
    {"$sort": {"total_reviews_written": -1}},
    {"$limit": 5}
]

print("--- 3. Top 5 Most Active Users ---")
result_3 = list(reviews_col.aggregate(pipeline_user_reviews))
for doc in result_3:
    print(doc)

--- 3. Top 5 Most Active Users ---
{'_id': 'A3OXHLG6DIBRW8', 'total_reviews_written': 8}
{'_id': 'A2LXX47A0KMJVX', 'total_reviews_written': 8}
{'_id': 'A3LGT6UZL99IW1', 'total_reviews_written': 7}
{'_id': 'A680RUE1FDO8B', 'total_reviews_written': 7}
{'_id': 'A32O5FZH994CNY', 'total_reviews_written': 6}


### Pipeline 4: Reviews Over Time (Temporal Analysis)
**Objective:** Track the volume of reviews being generated month over month.

**How it works:** 1. `$project`: Takes the `unixReviewTime` (which is in seconds), converts it to milliseconds, transforms it into a Date object, and formats it as a `YYYY-MM` string.
2. `$group`: Groups by this newly created `year_month` string to count the reviews.
3. `$sort`: Orders the results chronologically.

In [5]:
pipeline_time = [
    {"$project": {
        "year_month": {
            "$dateToString": {
                "format": "%Y-%m",
                "date": {"$toDate": {"$multiply": ["$unixReviewTime", 1000]}} 
            }
        }
    }},
    {"$group": {
        "_id": "$year_month", 
        "review_count": {"$sum": 1}
    }},
    {"$sort": {"_id": 1}} 
]

print("--- 4. Review Volume Over Time (First 10 months) ---")
result_4 = list(reviews_col.aggregate(pipeline_time))
for doc in result_4[:10]: 
    print(doc)

--- 4. Review Volume Over Time (First 10 months) ---
{'_id': '1999-12', 'review_count': 1}
{'_id': '2000-05', 'review_count': 2}
{'_id': '2000-06', 'review_count': 1}
{'_id': '2000-07', 'review_count': 1}
{'_id': '2000-09', 'review_count': 1}
{'_id': '2000-11', 'review_count': 1}
{'_id': '2000-12', 'review_count': 1}
{'_id': '2001-02', 'review_count': 2}
{'_id': '2001-04', 'review_count': 3}
{'_id': '2001-05', 'review_count': 1}


### Pipeline 5: Business Analysis (Verified vs. Unverified Sentiment)
**Objective:** Determine if users who have a "Verified Purchase" leave different ratings compared to unverified users.

**How it works:** 1. `$group`: Separates the reviews into two buckets based on the boolean `verified` field, calculating the average rating for both.
2. `$project`: Cleans up the output by renaming the boolean `_id` to readable strings ("Verified" / "Unverified") using `$cond`, and rounds the average rating to 2 decimal places.

In [6]:
pipeline_business = [
    {"$group": {
        "_id": "$verified",
        "avg_rating": {"$avg": "$overall"},
        "total_reviews": {"$sum": 1}
    }},
    {"$project": {
        "Purchase_Status": {"$cond": [{"$eq": ["$_id", True]}, "Verified", "Unverified"]},
        "avg_rating": {"$round": ["$avg_rating", 2]},
        "total_reviews": 1,
        "_id": 0
    }}
]

print("--- 5. Sentiment by Purchase Verification ---")
result_5 = list(reviews_col.aggregate(pipeline_business))
for doc in result_5:
    print(doc)

--- 5. Sentiment by Purchase Verification ---
{'total_reviews': 6918, 'Purchase_Status': 'Unverified', 'avg_rating': 3.94}
{'total_reviews': 60459, 'Purchase_Status': 'Verified', 'avg_rating': 4.3}
